# EOS Signal Universality Probe

**Theory validation experiment** — runs a single k=0 forward pass on a fully-masked canvas for each
of three diffusion LM architectures and plots the per-position EOS probability φⱼ.

**Proposition 1** predicts a CDF in all three models.
The geometry of that CDF is predicted to differ by training history:

| Model | Expected shape | Reason |
|---|---|---|
| LLaDA-8B-Instruct | Smooth sigmoid | From-scratch diffusion, no AR prior |
| Dream-v0-Instruct-7B | Spiky / AR-contaminated | Continual training from Qwen2.5, AR positional biases leak through |
| LLaDA 2.0 (mini) | Block staircase | Block diffusion coarsens boundary to block granularity |

**This notebook does three things in one figure:**
1. Validates universality of Proposition 1 (CDF exists on all architectures)
2. Explains why SmartCrop is benchmarked on LLaDA-8B (smoothest CDF, max-entropy estimator most appropriate)
3. Generates a concrete prediction about what model-specific estimators would look like

---
**How to run:** Execute cells sequentially. Each model loads, probes, saves `.npy` results, then unloads before the next.
A GPU is required. Run time is ~1–3 min per model for a 512-token canvas.

## 0. Config

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# ---- Experiment config ----
CANVAS_SIZE   = 512      # generation canvas (masked tokens)
N_EXAMPLES    = 4        # number of GSM8K prompts to average over
SEED          = 42
EOS_QUANTILE  = 0.9      # for marking the SmartCrop cutoff on plots
BLOCK_SIZE    = 32       # LLaDA 2.0 block size

# ---- Output directory ----
OUT_DIR = PROJECT_ROOT / "results" / "rebuttal" / "eos_universality"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Model IDs ----
MODELS = {
    "llada8b": {
        "hf_id":   "GSAI-ML/LLaDA-8B-Instruct",
        "label":   "LLaDA-8B-Instruct",
        "kind":    "llada",          # forward convention
        "color":   "#1f77b4",
    },
    "dream7b": {
        "hf_id":   "Dream-org/Dream-v0-Instruct-7B",
        "label":   "Dream-v0-Instruct-7B",
        "kind":    "dream",          # needs AR logit shift
        "color":   "#ff7f0e",
    },
    "llada2": {
        "hf_id":   "inclusionAI/LLaDA2.0-mini",
        "label":   "LLaDA-2.0-mini",
        "kind":    "llada2",         # block-diagonal attention mask
        "color":   "#2ca02c",
    },
}

print(f"Canvas: {CANVAS_SIZE} tokens | Examples: {N_EXAMPLES} | Out: {OUT_DIR}")

## 1. Load GSM8K prompts

We use a few GSM8K problems as representative prompts. Tokenization is model-specific (handled per model below).

In [ ]:
import random
from datasets import load_dataset

rng = random.Random(SEED)

# Load a small slice of GSM8K test split — no chat template applied yet
ds = load_dataset("openai/gsm8k", "main", split="test")
indices = rng.sample(range(len(ds)), N_EXAMPLES)
raw_prompts = [
    {"question": ds[i]["question"], "answer": ds[i]["answer"]}
    for i in indices
]

print(f"Loaded {len(raw_prompts)} prompts")
for i, p in enumerate(raw_prompts):
    print(f"  [{i}] {p['question'][:80]}...")

## 2. Helper functions

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F


# Fallback mask token IDs when the tokenizer doesn't expose mask_token_id
_MASK_ID_FALLBACKS = {
    "GSAI-ML/LLaDA-8B-Instruct":      126336,
    "inclusionAI/LLaDA2.0-mini":      156895,
}

def resolve_mask_id(tokenizer, hf_id: str | None = None) -> int:
    """Return mask_token_id, falling back to hardcoded values if the tokenizer leaves it unset."""
    mid = getattr(tokenizer, "mask_token_id", None)
    if mid is not None:
        return mid
    if hf_id and hf_id in _MASK_ID_FALLBACKS:
        mid = _MASK_ID_FALLBACKS[hf_id]
        print(f"  WARNING: mask_token_id not set by tokenizer — using fallback {mid}")
        return mid
    # Last resort: search vocab for common mask token strings
    for candidate in ["<mask>", "[MASK]", "<|mask|>", "<|mdm_mask|>", "[gMASK]"]:
        mid = tokenizer.convert_tokens_to_ids(candidate)
        if mid != tokenizer.unk_token_id:
            print(f"  WARNING: mask_token_id resolved from vocab: '{candidate}' → {mid}")
            return mid
    raise ValueError(
        f"Cannot resolve mask_token_id for {hf_id}. "
        "Set tokenizer.mask_token_id manually before calling probe functions."
    )


def build_llada2_block_attn(total_len: int, block_size: int, device) -> torch.Tensor:
    """Block-diagonal causal attention mask used by LLaDA 2.0.

    Causal across blocks, bidirectional within each block.
    Returns a [1, 1, total_len, total_len] log-domain mask (0 or -inf).
    """
    num_blocks = (total_len + block_size - 1) // block_size
    block_mask = torch.tril(torch.ones(num_blocks, num_blocks, device=device))
    attn = (
        block_mask
        .repeat_interleave(block_size, dim=0)
        .repeat_interleave(block_size, dim=1)
    )  # [padded_len, padded_len]
    attn = attn[:total_len, :total_len].unsqueeze(0).unsqueeze(0)  # [1, 1, T, T]
    return attn.log().to(torch.bfloat16)


@torch.no_grad()
def probe_llada(
    model, tokenizer, raw_prompts, canvas_size: int, device, hf_id: str | None = None
) -> tuple[np.ndarray, np.ndarray, list]:
    """k=0 EOS probe for LLaDA-8B (standard bidirectional MDLM)."""
    mask_id = resolve_mask_id(tokenizer, hf_id)
    eos_id  = tokenizer.eos_token_id

    all_eos_probs = []
    for ex in raw_prompts:
        messages = [{"role": "user", "content": ex["question"]}]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        prompt_ids = tokenizer.encode(text, add_special_tokens=False)
        prompt_len = len(prompt_ids)

        total = prompt_len + canvas_size
        x = torch.full((1, total), mask_id, dtype=torch.long, device=device)
        x[0, :prompt_len] = torch.tensor(prompt_ids, dtype=torch.long, device=device)
        attn_mask = torch.ones(1, total, dtype=torch.long, device=device)

        logits = model(input_ids=x, attention_mask=attn_mask).logits  # [1, T, V]
        gen_logits = logits[0, prompt_len:]   # [canvas, V]
        eos_probs  = F.softmax(gen_logits, dim=-1)[:, eos_id].cpu().float().numpy()
        all_eos_probs.append(eos_probs)

    mean_eos = np.mean(all_eos_probs, axis=0)
    mean_survival = np.cumprod(1.0 - mean_eos)
    return mean_eos, mean_survival, all_eos_probs


@torch.no_grad()
def probe_dream(
    model, tokenizer, raw_prompts, canvas_size: int, device, hf_id: str | None = None
) -> tuple[np.ndarray, np.ndarray, list]:
    """k=0 EOS probe for Dream-7B (AR-shifted bidirectional MDLM).

    Dream uses right-shift: model output at position i predicts token i+1.
    Apply shift before reading EOS probabilities.
    """
    mask_id = resolve_mask_id(tokenizer, hf_id)
    eos_id  = tokenizer.eos_token_id

    # Patch forward to auto-cast float32 attention mask to model dtype
    _orig_fwd = model.forward
    def _autocast_fwd(*args, **kwargs):
        if len(args) > 1 and isinstance(args[1], torch.Tensor):
            args = list(args)
            args[1] = args[1].to(next(model.parameters()).dtype)
            args = tuple(args)
        return _orig_fwd(*args, **kwargs)
    model.forward = _autocast_fwd

    all_eos_probs = []
    for ex in raw_prompts:
        messages = [{"role": "user", "content": ex["question"]}]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        prompt_ids = tokenizer.encode(text, add_special_tokens=False)
        prompt_len = len(prompt_ids)

        total = prompt_len + canvas_size
        x = torch.full((1, total), eos_id, dtype=torch.long, device=device)
        x[0, :prompt_len] = torch.tensor(prompt_ids, dtype=torch.long, device=device)
        x[0, prompt_len:] = mask_id

        logits_raw = model(x, "full", None).logits              # [1, T, V]
        # AR shift: logits[:, i] predicts token i+1 → shift right by 1
        logits_shifted = torch.cat(
            [logits_raw[:, :1], logits_raw[:, :-1]], dim=1
        )  # [1, T, V]
        gen_logits = logits_shifted[0, prompt_len:]             # [canvas, V]
        eos_probs  = F.softmax(gen_logits, dim=-1)[:, eos_id].cpu().float().numpy()
        all_eos_probs.append(eos_probs)

    mean_eos = np.mean(all_eos_probs, axis=0)
    mean_survival = np.cumprod(1.0 - mean_eos)
    return mean_eos, mean_survival, all_eos_probs


@torch.no_grad()
def probe_llada2(
    model, tokenizer, raw_prompts, canvas_size: int, block_size: int, device,
    hf_id: str | None = None
) -> tuple[np.ndarray, np.ndarray, list]:
    """k=0 EOS probe for LLaDA 2.0 (block diffusion).

    Uses the block-diagonal causal attention mask from generation.
    EOS logits within a block are identical → expected staircase pattern.
    """
    mask_id = resolve_mask_id(tokenizer, hf_id)
    eos_id  = tokenizer.eos_token_id

    all_eos_probs = []
    for ex in raw_prompts:
        messages = [{"role": "user", "content": ex["question"]}]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        prompt_ids = tokenizer.encode(text, add_special_tokens=False)
        prompt_len = len(prompt_ids)

        raw_total = prompt_len + canvas_size
        num_blocks = (raw_total + block_size - 1) // block_size
        total = num_blocks * block_size

        x = torch.full((1, total), mask_id, dtype=torch.long, device=device)
        x[0, :prompt_len] = torch.tensor(prompt_ids, dtype=torch.long, device=device)

        block_attn   = build_llada2_block_attn(total, block_size, device)  # [1,1,T,T]
        position_ids = torch.arange(total, device=device).unsqueeze(0)     # [1, T]

        logits = model(
            input_ids=x,
            attention_mask=block_attn,
            position_ids=position_ids,
        ).logits  # [1, T, V]

        gen_logits = logits[0, prompt_len:prompt_len + canvas_size]  # [canvas, V]
        eos_probs  = F.softmax(gen_logits, dim=-1)[:, eos_id].cpu().float().numpy()
        all_eos_probs.append(eos_probs)

    mean_eos = np.mean(all_eos_probs, axis=0)
    mean_survival = np.cumprod(1.0 - mean_eos)
    return mean_eos, mean_survival, all_eos_probs


def find_tau(survival: np.ndarray, quantile: float = 0.9) -> int | None:
    """First position where CDF = 1 - S(t) crosses quantile."""
    cdf  = 1.0 - survival
    hits = np.where(cdf >= quantile)[0]
    return int(hits[0]) + 1 if len(hits) > 0 else None


def save_results(key: str, eos_probs, survival, all_eos_probs):
    np.save(OUT_DIR / f"{key}_eos_probs.npy",  eos_probs)
    np.save(OUT_DIR / f"{key}_survival.npy",   survival)
    np.save(OUT_DIR / f"{key}_all_examples.npy", np.array(all_eos_probs))
    print(f"  Saved {key} results to {OUT_DIR}")


print("Helpers defined.")

## 3. Probe — LLaDA-8B-Instruct

In [ ]:
import gc
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

cfg = MODELS["llada8b"]
print(f"Loading {cfg['hf_id']} ...")

tokenizer_llada = AutoTokenizer.from_pretrained(cfg["hf_id"], trust_remote_code=True)
model_llada     = AutoModel.from_pretrained(
    cfg["hf_id"], torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
).eval()
model_llada.requires_grad_(False)

print(f"  mask_token_id={tokenizer_llada.mask_token_id}  eos_token_id={tokenizer_llada.eos_token_id}")

eos_llada, surv_llada, all_llada = probe_llada(
    model_llada, tokenizer_llada, raw_prompts, CANVAS_SIZE, device, hf_id=cfg["hf_id"]
)
tau_llada = find_tau(surv_llada, EOS_QUANTILE)
print(f"  τ₀.₉ (SmartCrop cutoff) = {tau_llada} tokens | peak EOS prob = {eos_llada.max():.4f}")

save_results("llada8b", eos_llada, surv_llada, all_llada)

del model_llada
gc.collect()
torch.cuda.empty_cache()
print("LLaDA-8B unloaded.")

## 4. Probe — Dream-v0-Instruct-7B

**Note on EOS semantics:** Dream is continually trained from Qwen2.5. If Dream does not use
EOS-padded training (i.e., the generation region is not trained with EOS as a termination signal),
Proposition 1 explicitly predicts *no signal* here. A flat or nonsensical curve is a valid negative
result that strengthens the paper — it shows the signal is specific to EOS-padded training.

In [ ]:
cfg = MODELS["dream7b"]
print(f"Loading {cfg['hf_id']} ...")

tokenizer_dream = AutoTokenizer.from_pretrained(cfg["hf_id"], trust_remote_code=True)
model_dream     = AutoModel.from_pretrained(
    cfg["hf_id"], torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
).eval()
model_dream.requires_grad_(False)

print(f"  mask_token_id={tokenizer_dream.mask_token_id}  eos_token_id={tokenizer_dream.eos_token_id}")

eos_dream, surv_dream, all_dream = probe_dream(
    model_dream, tokenizer_dream, raw_prompts, CANVAS_SIZE, device, hf_id=cfg["hf_id"]
)
tau_dream = find_tau(surv_dream, EOS_QUANTILE)
print(f"  τ₀.₉ = {tau_dream} tokens | peak EOS prob = {eos_dream.max():.4f}")

save_results("dream7b", eos_dream, surv_dream, all_dream)

del model_dream
gc.collect()
torch.cuda.empty_cache()
print("Dream-7B unloaded.")

## 5. Probe — LLaDA-2.0-mini

LLaDA 2.0 uses block diffusion: a block-diagonal causal attention mask where each block sees
all previous blocks but positions within the same block are bidirectionally attended.
The prediction: EOS logits within a block are identical → staircase at block boundaries.

In [ ]:
cfg = MODELS["llada2"]
print(f"Loading {cfg['hf_id']} ...")

try:
    import dllm.pipelines.llada2  # registers LLaDA2MoeConfig with AutoConfig
    print("  dllm.pipelines.llada2 registered")
except ImportError:
    print("  WARNING: dllm not found — AutoModel may fail for LLaDA 2.0")

tokenizer_llada2 = AutoTokenizer.from_pretrained(cfg["hf_id"], trust_remote_code=True)
model_llada2     = AutoModel.from_pretrained(
    cfg["hf_id"], torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
).eval()
model_llada2.requires_grad_(False)

print(f"  mask_token_id={tokenizer_llada2.mask_token_id}  eos_token_id={tokenizer_llada2.eos_token_id}")
print(f"  block_size={BLOCK_SIZE}")

eos_llada2, surv_llada2, all_llada2 = probe_llada2(
    model_llada2, tokenizer_llada2, raw_prompts, CANVAS_SIZE, BLOCK_SIZE, device,
    hf_id=cfg["hf_id"]
)
tau_llada2 = find_tau(surv_llada2, EOS_QUANTILE)
print(f"  τ₀.₉ = {tau_llada2} tokens | peak EOS prob = {eos_llada2.max():.4f}")

save_results("llada2", eos_llada2, surv_llada2, all_llada2)

del model_llada2
gc.collect()
torch.cuda.empty_cache()
print("LLaDA-2.0-mini unloaded.")

## 6. Load saved results and plot

Run this cell independently to re-plot without re-running models.

In [ ]:
import numpy as np
from pathlib import Path

OUT_DIR = Path("..").resolve() / "results" / "rebuttal" / "eos_universality"

def load_results(key: str):
    eos      = np.load(OUT_DIR / f"{key}_eos_probs.npy")
    survival = np.load(OUT_DIR / f"{key}_survival.npy")
    all_ex   = np.load(OUT_DIR / f"{key}_all_examples.npy")
    return eos, survival, all_ex

results = {}
for key, cfg in MODELS.items():
    try:
        eos, surv, all_ex = load_results(key)
        results[key] = {"eos": eos, "survival": surv, "all": all_ex, **cfg}
        print(f"Loaded {cfg['label']}: canvas={len(eos)}, peak_eos={eos.max():.4f}")
    except FileNotFoundError:
        print(f"WARNING: {key} results not found — run probe cells first")

print(f"\nLoaded {len(results)}/3 models")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

EOS_QUANTILE = 0.9
CANVAS_SIZE  = len(next(iter(results.values()))["eos"])
positions    = np.arange(CANVAS_SIZE)

# --- Expected-shape annotations ---
EXPECTED_SHAPES = {
    "llada8b": "smooth sigmoid\n(no AR prior)",
    "dream7b": "spiky / AR-contaminated\n(Qwen2.5 positional biases)",
    "llada2":  f"block staircase\n(block_size={BLOCK_SIZE} granularity)",
}

n_models = len(results)
fig = plt.figure(figsize=(6.5 * n_models, 11))
gs  = gridspec.GridSpec(3, n_models, height_ratios=[1.2, 1.2, 0.9], hspace=0.45, wspace=0.35)

for col, (key, data) in enumerate(results.items()):
    eos      = data["eos"]
    survival = data["survival"]
    all_ex   = data["all"]
    color    = data["color"]
    label    = data["label"]

    cdf = 1.0 - survival
    hits = np.where(cdf >= EOS_QUANTILE)[0]
    tau  = int(hits[0]) + 1 if len(hits) > 0 else None

    # ── Row 0: EOS probability φⱼ per position ──
    ax0 = fig.add_subplot(gs[0, col])
    # Individual examples in light color
    for ex in all_ex:
        ax0.plot(positions, ex, color=color, lw=0.5, alpha=0.25)
    # Mean in bold
    ax0.plot(positions, eos, color=color, lw=1.8, label="mean")
    ax0.set_title(f"Panel {'ABC'[col]} — {label}\n{EXPECTED_SHAPES.get(key, '')}",
                  fontsize=10, fontweight="bold")
    ax0.set_xlabel("Token position j", fontsize=9)
    ax0.set_ylabel("φⱼ = P(EOS at j | all-masked canvas)", fontsize=8)
    ax0.set_xlim(0, CANVAS_SIZE)
    ax0.set_ylim(-0.02, min(1.05, eos.max() * 1.3 + 0.05))
    ax0.grid(True, alpha=0.2)
    ax0.legend(fontsize=8)

    # ── Row 1: Survival S(t) = CDF complement ──
    ax1 = fig.add_subplot(gs[1, col])
    ax1.plot(positions, survival, color=color, lw=2.0, label="S(t) = ∏(1−φⱼ)")
    ax1.axhline(1 - EOS_QUANTILE, color="red", ls="--", lw=1.2,
                label=f"S={1-EOS_QUANTILE:.1f} ({int(EOS_QUANTILE*100)}th-pct cutoff)")
    if tau is not None:
        ax1.axvline(tau, color="red", ls=":", lw=1.5, alpha=0.8)
        ax1.annotate(
            f"τ₀.₉={tau}",
            xy=(tau, 1 - EOS_QUANTILE),
            xytext=(tau + CANVAS_SIZE * 0.04, 0.35),
            fontsize=8.5, color="red",
            arrowprops=dict(arrowstyle="->", color="red", lw=0.8),
        )
    ax1.set_xlabel("Token position t", fontsize=9)
    ax1.set_ylabel("S(t) = P(not ended by t)", fontsize=8)
    ax1.set_xlim(0, CANVAS_SIZE)
    ax1.set_ylim(-0.02, 1.05)
    ax1.grid(True, alpha=0.2)
    ax1.legend(fontsize=8)

    # ── Row 2: CDF = 1 − S(t) ──
    ax2 = fig.add_subplot(gs[2, col])
    ax2.plot(positions, cdf, color=color, lw=2.0, label="CDF = 1 − S(t)")
    ax2.axhline(EOS_QUANTILE, color="red", ls="--", lw=1.2,
                label=f"q={EOS_QUANTILE} (SmartCrop cutoff)")
    if tau is not None:
        ax2.axvline(tau, color="red", ls=":", lw=1.5, alpha=0.8)
    ax2.set_xlabel("Token position t", fontsize=9)
    ax2.set_ylabel("CDF = P(ended by t)", fontsize=8)
    ax2.set_xlim(0, CANVAS_SIZE)
    ax2.set_ylim(-0.02, 1.05)
    ax2.grid(True, alpha=0.2)
    ax2.legend(fontsize=8)

fig.suptitle(
    "EOS Signal Universality — k=0 probe on fully-masked canvas\n"
    f"Dataset: GSM8K  |  Canvas: {CANVAS_SIZE} tokens  |  N={len(next(iter(results.values()))['all'])} examples",
    fontsize=12, fontweight="bold", y=1.01,
)

out_path = OUT_DIR / "eos_universality_3panel.pdf"
fig.savefig(out_path, dpi=200, bbox_inches="tight")
fig.savefig(str(out_path).replace(".pdf", ".png"), dpi=200, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

## 7. Per-example overlay (optional diagnostic)

In [ ]:
import matplotlib.cm as cm

# One row per model, one curve per example
fig, axes = plt.subplots(len(results), 1, figsize=(14, 4.5 * len(results)), sharex=True)
if len(results) == 1:
    axes = [axes]

for ax, (key, data) in zip(axes, results.items()):
    all_ex = data["all"]
    color  = data["color"]
    n_ex   = len(all_ex)
    ex_colors = cm.tab10(np.linspace(0, 0.9, n_ex))

    for i, (ex, ec) in enumerate(zip(all_ex, ex_colors)):
        surv_ex = np.cumprod(1.0 - ex)
        ax.plot(positions, surv_ex, color=ec, lw=1.2, alpha=0.85, label=f"ex {i}")

    # Mean survival
    ax.plot(positions, data["survival"], color="black", lw=2.2, ls="--", label="mean S(t)")
    ax.axhline(1 - EOS_QUANTILE, color="red", ls=":", lw=1.2)
    ax.set_ylabel("S(t)", fontsize=9)
    ax.set_title(f"{data['label']} — per-example survival curves", fontsize=10)
    ax.set_xlim(0, CANVAS_SIZE)
    ax.set_ylim(-0.02, 1.05)
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=8, ncol=min(n_ex + 1, 6), loc="upper right")

axes[-1].set_xlabel("Token position t", fontsize=9)
fig.suptitle("Per-example survival functions — variability across GSM8K prompts",
             fontsize=11, fontweight="bold")
fig.tight_layout()

out_path2 = OUT_DIR / "eos_universality_per_example.png"
fig.savefig(out_path2, dpi=150, bbox_inches="tight")
print(f"Saved: {out_path2}")
plt.show()

## 8. Summary statistics

In [ ]:
print("=" * 70)
print(f"{'Model':<30} {'τ₀.₉':>8} {'peak φ':>10} {'signal?':>10}")
print("-" * 70)

for key, data in results.items():
    eos  = data["eos"]
    surv = data["survival"]
    cdf  = 1.0 - surv
    hits = np.where(cdf >= EOS_QUANTILE)[0]
    tau  = int(hits[0]) + 1 if len(hits) > 0 else None
    peak = float(eos.max())
    has_signal = peak > 0.01  # non-flat EOS probability
    print(f"{data['label']:<30} {str(tau):>8} {peak:>10.4f} {'YES' if has_signal else 'NO (flat)':>10}")

print("=" * 70)
print()
print("Interpretation:")
print("  - τ₀.₉: SmartCrop 90th-percentile length prediction at k=0")
print("  - peak φ: max per-position EOS probability (>0.01 indicates a CDF signal)")
print("  - 'flat' result for Dream is a valid negative — confirms signal")
print("    requires EOS-padded training, not a generic MDLM property")

## 9. Interpretation guide

### What to look for

**Panel A (LLaDA-8B):** The theory predicts a smooth sigmoid-shaped CDF. The EOS probability φⱼ
should start near 0, rise monotonically, and reach a plateau. This is the architecture where
the max-entropy estimator (product formula) is most appropriate: the independence assumption
holds because the from-scratch diffusion training produces independent positional marginals.

**Panel B (Dream-7B):** Two sub-cases:
- *Signal present but spiky:* The CDF exists but marginals carry AR artifacts from Qwen2.5's
  positional encoding. Inter-position correlations violate the independence assumption more
  severely than in LLaDA-8B.
- *No signal (flat φⱼ ≈ 0):* Dream was not trained with EOS padding; Proposition 1 then
  predicts *no* signal, which is a valid negative result confirming the theory.

**Panel C (LLaDA 2.0):** Expect a staircase pattern — EOS logits are identical within
each block of size `BLOCK_SIZE` because the block-diagonal attention mask prevents positions
within a block from distinguishing each other. SmartCrop on LLaDA 2.0 would predict lengths
at block granularity, not token granularity.

### What this figure proves

1. **Universality of Proposition 1:** All models with EOS-padded training show a CDF.
2. **Architecture shapes geometry:** The CDF geometry encodes training history.
3. **SmartCrop domain:** LLaDA-8B is the architecture where SmartCrop is most natural.
   Porting to Dream or LLaDA 2.0 requires model-specific modifications.
4. **Concrete predictions:** Falsifiable predictions about what shape to expect on each
   architecture — this is not cherry-picking, it's theory confirmation.